# Multi-Task U-Net: Footprint + Altura Condicional (Bins em Metros)
Implementação enxuta da **Sugestão 2** sobre a arquitetura base vencedora (`BEST_SPEC`):
- **Cabeça 1 (`footprint_prob`):** segmentação binária de edificação ($p_{\text{bldg}} \in [0, 1]$, perda `BCE + Dice`), separando solo/rua de prédio.
- **Cabeça 2 (`height_m`):** regressão ordinal por distribuição discreta em $K=32$ *bins* log-espaçados (0 a $150\text{ m}$), treinada **exclusivamente nos pixels com prédio (`y > 0`)** via *Cross-Entropy* + *RMSE* direto em metros (eliminando o viés negativo e a explosão de `expm1`).
- **Predição final:** $\hat{h} = p_{\text{bldg}} \cdot \hat{h}_{\text{cond}}$ (com opção de limiar de máscara).

## 1. Setup, Dados e Carregamento da Arquitetura Base

In [ ]:
# Setup: paths and shared folds (cnn/cnn_setup.py). Run this notebook from the repository root or from cnn/notebooks.
import sys, pathlib
for _p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent, pathlib.Path.cwd() / "cnn"):
    if (_p / "cnn_setup.py").exists():
        sys.path.insert(0, str(_p)); break
import cnn_setup as cs

In [ ]:
import ctypes, gc, glob, json, os, zlib
import numpy as np, pandas as pd, tensorflow as tf, mlflow
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

for g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)

TRAIN_DIR = str(cs.TRAIN_DIR)
PATTERN = "*.tfrecord.gz"
C_IN, C_OUT, S_GRID, NODATA = 48, 3, 33, -9999.0
PX = 33
K_FOLDS, OUTER_FOLD = 5, 0
INNER_VAL_FOLD = (OUTER_FOLD + 1) % K_FOLDS
BATCH, EPOCHS, LR, DROPOUT = 64, 100, 1e-3, 0.3

# Configuração dos bins de altura (em metros) para a cabeça condicional
N_BINS = 32
H_MAX_BIN = 150.0
BIN_EDGES = np.geomspace(1.0, H_MAX_BIN + 1.0, N_BINS + 1) - 1.0  # 0 a 150 m com maior resolução nos prédios baixos/médios
BIN_CENTERS = 0.5 * (BIN_EDGES[:-1] + BIN_EDGES[1:])
BIN_CENTERS_TF = tf.constant(BIN_CENTERS, dtype=tf.float32)

PAPER_BINS = [-np.inf, 2, 6, 15, 30, np.inf]
PAPER_NAMES = ["h_lt2", "h_2_6", "h_6_15", "h_15_30", "h_gt30"]

# Herda automaticamente a melhor arquitetura encontrada anteriormente (ou usa u32_c2 como fallback)
BEST_SPEC = {"name": "u32_c2", "filters": [32, 64, 128], "convs": 2, "se": False, "tatt": False}
cfg_candidates = sorted(glob.glob(str(cs.OUT / "runs_cnn_iterative" / "*" / "config.json")), key=os.path.getmtime)
if cfg_candidates:
    with open(cfg_candidates[-1], "r", encoding="utf-8") as f:
        prev = json.load(f)
    if "best_spec" in prev:
        BEST_SPEC = prev["best_spec"]
print("Arquitetura base:", BEST_SPEC)

## 2. Leitura, Pré-processamento e Alvo Multi-Cabeça (`[h_metros, mask_valido, mask_predio]`)

In [ ]:
spec_tfr = {
    "x": tf.io.FixedLenFeature([C_IN * S_GRID * S_GRID], tf.float32),
    "y": tf.io.FixedLenFeature([C_OUT * S_GRID * S_GRID], tf.float32),
    "block_x": tf.io.FixedLenFeature([1], tf.float32),
    "block_y": tf.io.FixedLenFeature([1], tf.float32),
    "sample_id": tf.io.FixedLenFeature([1], tf.string),
    "city": tf.io.FixedLenFeature([1], tf.string),
    "split": tf.io.FixedLenFeature([1], tf.string),
}

def parse_fn(rec):
    c, h = S_GRID // 2, PX // 2
    e = tf.io.parse_single_example(rec, spec_tfr)
    x = tf.transpose(tf.reshape(e["x"], [C_IN, S_GRID, S_GRID]), [1, 2, 0])[c - h:c + h + 1, c - h:c + h + 1]
    y = tf.transpose(tf.reshape(e["y"], [C_OUT, S_GRID, S_GRID]), [1, 2, 0])[c - h:c + h + 1, c - h:c + h + 1]
    valid = tf.cast(y[..., 0] != NODATA, tf.float32)
    bldg_h = tf.where(y[..., 1] == NODATA, 0.0, tf.maximum(y[..., 1], 0.0))
    return x, bldg_h, valid, e["block_x"][0], e["block_y"][0], e["sample_id"][0], e["city"][0], e["split"][0]

files = sorted(glob.glob(f"{TRAIN_DIR}/{PATTERN}", recursive=True))
ds = tf.data.TFRecordDataset(files, compression_type="GZIP", num_parallel_reads=tf.data.AUTOTUNE)
ds = ds.map(parse_fn, num_parallel_calls=tf.data.AUTOTUNE).filter(lambda *a: tf.equal(a[7], "train"))
cols = [np.concatenate(c) for c in zip(*ds.batch(512).prefetch(1).as_numpy_iterator())]
d = dict(zip(["x", "h_map", "mask_px", "bx", "by", "sid", "city", "split"], cols))
for k in ("sid", "city", "split"):
    d[k] = np.array([s.decode() for s in d[k]])
_, keep = np.unique(d["sid"], return_index=True)
d = {k: v[np.sort(keep)] for k, v in d.items()}

folds = cs.fold_of(d["city"], d["bx"], d["by"], K_FOLDS)
tr = {k: v[~np.isin(folds, [OUTER_FOLD, INNER_VAL_FOLD])] for k, v in d.items()}
va = {k: v[folds == INNER_VAL_FOLD] for k, v in d.items()}
ho = {k: v[folds == OUTER_FOLD] for k, v in d.items()}
d = None; gc.collect()

VALID_CH = [s * 12 + b for s in range(4) for b in (0, 2)]

def channel_stats(X, chunk=1024):
    n = np.zeros(X.shape[-1]); s = np.zeros(X.shape[-1]); s2 = np.zeros(X.shape[-1])
    for i in range(0, len(X), chunk):
        xb = X[i:i + chunk].astype("float64")
        v = xb != NODATA; xb = np.where(v, xb, 0.0)
        n += v.sum((0, 1, 2)); s += xb.sum((0, 1, 2)); s2 += (xb ** 2).sum((0, 1, 2))
    mean = s / n.clip(1)
    return mean.astype("float32"), np.sqrt((s2 / n.clip(1) - mean ** 2).clip(0)).clip(1e-6).astype("float32")

def prep(X, mean, std, chunk=1024):
    n, c = len(X), X.shape[3]
    arr = np.empty((n, X.shape[1], X.shape[2], c + len(VALID_CH)), dtype="float32")
    for s in range(0, n, chunk):
        xb = X[s:s + chunk]; valid = xb != NODATA; e = s + len(xb)
        arr[s:e, ..., :c] = np.where(valid, (xb - mean) / std, 0.0)
        arr[s:e, ..., c:] = valid[..., VALID_CH]
    return arr

def pack_multitask(h_map, mask_valid):
    # Canal 0: altura real em metros | Canal 1: pixel válido | Canal 2: máscara binária de prédio (y > 0)
    is_bldg = ((h_map > 0.0) & (mask_valid > 0.5)).astype("float32")
    return np.stack([h_map.astype("float32"), mask_valid.astype("float32"), is_bldg], axis=-1)

mean, std = channel_stats(tr["x"])
Xt, Yt = prep(tr["x"], mean, std), pack_multitask(tr["h_map"], tr["mask_px"])
Xv, Yv = prep(va["x"], mean, std), pack_multitask(va["h_map"], va["mask_px"])
Xho = prep(ho["x"], mean, std)
hv, mv = va["h_map"].astype("float32"), va["mask_px"].astype("float32")
hho, mho = ho["h_map"].astype("float32"), ho["mask_px"].astype("float32")
tr = va = ho = None; gc.collect()

def make_ds(X, Y, batch, training):
    n_real = len(X)
    plan = np.concatenate([np.stack([np.arange(n_real), np.full(n_real, k)], 1) for k in (0, 1, 2, 3)]) if training else np.stack([np.arange(n_real), np.zeros(n_real, int)], 1)
    rng = np.random.default_rng(SEED)
    def gen():
        order = rng.permutation(len(plan)) if training else np.arange(len(plan))
        for s in range(0, len(plan), batch):
            p = plan[order[s:s + batch]]
            xs, ys = [], []
            for rt in np.unique(p[:, 1]):
                sel = p[p[:, 1] == rt, 0]
                xb, yb = X[sel], Y[sel]
                if rt:
                    xb, yb = np.rot90(xb, rt, axes=(1, 2)), np.rot90(yb, rt, axes=(1, 2))
                xs.append(np.ascontiguousarray(xb)); ys.append(np.ascontiguousarray(yb))
            yield np.concatenate(xs), np.concatenate(ys)
    sig = (tf.TensorSpec((None, *X.shape[1:]), tf.float32), tf.TensorSpec((None, *Y.shape[1:]), tf.float32))
    return tf.data.Dataset.from_generator(gen, output_signature=sig).prefetch(2)

print(f"Treino: {Xt.shape} | Val: {Xv.shape} | Hold-out: {Xho.shape}")

## 3. Arquitetura U-Net Multi-Cabeça (`footprint_prob` + `height_bins`) e Perdas Compostas

In [ ]:
def conv_block(x, f):
    x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.ReLU()(x)

def se_block(x, r=8):
    c = x.shape[-1]
    s = layers.GlobalAveragePooling2D(keepdims=True)(x)
    s = layers.Dense(max(c // r, 4), activation="relu")(s)
    s = layers.Dense(c, activation="sigmoid")(s)
    return layers.Multiply()([x, s])

def build_multitask_unet(spec, input_shape, n_bins=N_BINS, dropout=DROPOUT):
    h, w, _ = input_shape
    filters = spec["filters"]
    convs = spec.get("convs", 2)
    se = spec.get("se", False)
    depth = len(filters) - 1
    p = (-h) % (2 ** depth)

    inp = layers.Input(input_shape)
    x = layers.ZeroPadding2D(((0, p), (0, p)))(inp) if p else inp

    skips = []
    for f in filters[:-1]:
        for _ in range(convs):
            x = conv_block(x, f)
        if se:
            x = se_block(x)
        skips.append(x)
        x = layers.MaxPool2D()(x)

    x = conv_block(x, filters[-1])
    if se:
        x = se_block(x)
    x = layers.Dropout(dropout)(x)

    for f, sk in zip(filters[-2::-1], skips[::-1]):
        x = layers.UpSampling2D()(x)
        x = layers.Concatenate()([x, sk])
        for _ in range(convs):
            x = conv_block(x, f)
        if se:
            x = se_block(x)

    if p:
        x = layers.Cropping2D(((0, p), (0, p)))(x)

    # Cabeça 1: Probabilidade de ser edificação (footprint)
    b1 = conv_block(x, filters[0])
    footprint_prob = layers.Conv2D(1, 1, activation="sigmoid", name="footprint_prob")(b1)

    # Cabeça 2: Distribuição discreta de altura condicional (N_BINS) -> altura esperada em metros
    b2 = conv_block(x, filters[0])
    bin_logits = layers.Conv2D(n_bins, 1, name="bin_logits")(b2)
    bin_probs = layers.Softmax(axis=-1, name="bin_probs")(bin_logits)
    cond_height_m = layers.Lambda(
        lambda pr: tf.reduce_sum(pr * BIN_CENTERS_TF, axis=-1, keepdims=True),
        name="cond_height_m"
    )(bin_probs)

    # Saída empacotada [footprint_prob (1), cond_height_m (1), bin_probs (N_BINS)]
    out_pack = layers.Concatenate(axis=-1, name="multitask_out")([footprint_prob, cond_height_m, bin_probs])
    return keras.Model(inp, out_pack)

BIN_EDGES_TF = tf.constant(BIN_EDGES[1:-1], dtype=tf.float32)

def multitask_loss(y_true, y_pred):
    h_true = y_true[..., 0]
    m_valid = y_true[..., 1]
    m_bldg = y_true[..., 2]

    p_bldg = tf.clip_by_value(y_pred[..., 0], 1e-6, 1.0 - 1e-6)
    h_cond = y_pred[..., 1]
    bin_probs = tf.clip_by_value(y_pred[..., 2:], 1e-7, 1.0)

    # 1) Perda de segmentação do footprint (BCE + Soft Dice) nos pixels válidos
    bce = -(m_bldg * tf.math.log(p_bldg) + (1.0 - m_bldg) * tf.math.log(1.0 - p_bldg))
    loss_bce = tf.reduce_sum(bce * m_valid) / (tf.reduce_sum(m_valid) + 1e-8)

    inter = tf.reduce_sum(p_bldg * m_bldg * m_valid)
    union = tf.reduce_sum((p_bldg + m_bldg) * m_valid)
    loss_dice = 1.0 - (2.0 * inter + 1e-6) / (union + 1e-6)
    loss_seg = 0.5 * loss_bce + 0.5 * loss_dice

    # 2) Perda condicional de altura (APENAS onde m_bldg == 1): Cross-Entropy nos bins + RMSE em metros
    bin_idx = tf.raw_ops.Bucketize(input=tf.clip_by_value(h_true, 0.0, H_MAX_BIN - 1e-3), boundaries=list(BIN_EDGES[1:-1]))
    y_onehot = tf.one_hot(bin_idx, depth=N_BINS, dtype=tf.float32)
    ce_bins = -tf.reduce_sum(y_onehot * tf.math.log(bin_probs), axis=-1)
    loss_ce_cond = tf.reduce_sum(ce_bins * m_bldg) / (tf.reduce_sum(m_bldg) + 1e-8)

    sq_cond = tf.square(h_true - h_cond)
    loss_rmse_cond = tf.sqrt(tf.reduce_sum(sq_cond * m_bldg) / (tf.reduce_sum(m_bldg) + 1e-8) + 1e-8)

    # 3) Consistência global (h_hat = p_bldg * h_cond) em todos os pixels válidos
    h_hat = p_bldg * h_cond
    loss_rmse_all = tf.sqrt(tf.reduce_sum(tf.square(h_true - h_hat) * m_valid) / (tf.reduce_sum(m_valid) + 1e-8) + 1e-8)

    return 2.0 * loss_seg + 0.5 * loss_ce_cond + 0.5 * loss_rmse_cond + 0.5 * loss_rmse_all

def val_rmse_combined_m(y_true, y_pred):
    h_true, m_valid = y_true[..., 0], y_true[..., 1]
    h_hat = y_pred[..., 0] * y_pred[..., 1]
    return tf.sqrt(tf.reduce_sum(tf.square(h_true - h_hat) * m_valid) / (tf.reduce_sum(m_valid) + 1e-8) + 1e-8)

model = build_multitask_unet(BEST_SPEC, Xt.shape[1:])
model.compile(optimizer=keras.optimizers.Adam(LR), loss=multitask_loss, metrics=[val_rmse_combined_m])
print(f"Parâmetros do modelo multi-cabeça: {model.count_params():,}")

## 4. Treinamento e Avaliação (Validação Interna e Hold-Out)

In [ ]:
os.makedirs(cs.OUT / "multitask_experimentation", exist_ok=True)
ckpt_path = f"{cs.OUT}/multitask_experimentation/best_multitask.keras"
FORCE_RETRAIN = False  # mude para True só se quiser treinar do zero

if not FORCE_RETRAIN and os.path.exists(ckpt_path):
    print(f"Checkpoint encontrado — pulando treino e carregando pesos de: {ckpt_path}")
    model.load_weights(ckpt_path)
else:
    cbs = [
        keras.callbacks.EarlyStopping(monitor="val_val_rmse_combined_m", mode="min", patience=15, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_val_rmse_combined_m", mode="min", factor=0.5, patience=6, min_lr=1e-5),
        keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_val_rmse_combined_m", mode="min", save_best_only=True),
    ]
    hist = model.fit(
        make_ds(Xt, Yt, BATCH, True),
        validation_data=make_ds(Xv, Yv, BATCH, False),
        epochs=EPOCHS,
        callbacks=cbs,
        verbose=1,
    )
def predict_multitask(m, X, batch=BATCH, prob_thresh=None):
    preds = np.concatenate([m.predict_on_batch(X[s:s + batch]) for s in range(0, len(X), batch)], axis=0)
    p_bldg = preds[..., 0]
    h_cond = preds[..., 1]
    if prob_thresh is not None:
        h_final = np.where(p_bldg >= prob_thresh, h_cond, 0.0)
    else:
        h_final = p_bldg * h_cond
    return p_bldg, h_cond, np.clip(h_final, 0.0, None)

def dense_metrics(h_true, h_pred, mask):
    valid = mask > 0.5
    y, p = h_true[valid].astype("float64"), h_pred[valid].astype("float64")
    bld = y > 0
    err = p - y
    err_b = p[bld] - y[bld]
    idx = np.digitize(y, PAPER_BINS[1:-1], right=True)
    cls_rmse = {f"RMSE_{name}": float(np.sqrt(((p[idx == k] - y[idx == k]) ** 2).mean()))
                for k, name in enumerate(PAPER_NAMES) if (idx == k).any()}
    return {
        "macro_RMSE_px": float(np.mean(list(cls_rmse.values()))),
        "RMSE_px": float(np.sqrt((err ** 2).mean())),
        "MAE_px": float(np.abs(err).mean()),
        "bias_px": float(err.mean()),
        "R2_px": float(1.0 - (err ** 2).sum() / (((y - y.mean()) ** 2).sum() + 1e-8)),
        "RMSE_bldg_px": float(np.sqrt((err_b ** 2).mean())),
        "MAE_bldg_px": float(np.abs(err_b).mean()),
        **cls_rmse,
    }

# Otimiza limiar de probabilidade do footprint na validação (suave vs limiarizado)
p_va, hcond_va, hsoft_va = predict_multitask(model, Xv, prob_thresh=None)
best_th, best_rmse = None, dense_metrics(hv, hsoft_va, mv)["RMSE_px"]
for th in [0.35, 0.45, 0.50, 0.55]:
    r_th = dense_metrics(hv, np.where(p_va >= th, hcond_va, 0.0), mv)["RMSE_px"]
    if r_th < best_rmse:
        best_rmse, best_th = r_th, th

_, _, hpred_va = predict_multitask(model, Xv, prob_thresh=best_th)
_, _, hpred_ho = predict_multitask(model, Xho, prob_thresh=best_th)

print(f"Melhor modo de combinação (val): {'suave (p*h)' if best_th is None else f'limiar p>={best_th}'}")
pd.DataFrame({
    "Validacao_MultiTask": dense_metrics(hv, hpred_va, mv),
    "HoldOut_MultiTask": dense_metrics(hho, hpred_ho, mho),
}).T.round(4)

In [ ]:
# ==============================================================================
# 5. RECONSTRUCAO ESPACIAL DE CIDADE COMPLETA (ESTILO LIDAR NOS PATCHES SELECIONADOS)
# ==============================================================================
import matplotlib.pyplot as plt

TARGET_CITY = "porto_alegre"   # Ex.: 'porto_alegre', 'sao_paulo', 'rio_de_janeiro', 'belo_horizonte', 'curitiba', 'niteroi', 'recife', 'salvador'
INCLUDE_TEST_FOLDER = False    # True para incluir tambem patch_dataset_test/*.tfrecord.gz
SCALE_M = 10.0

spec_spatial = {
    "x": tf.io.FixedLenFeature([C_IN * S_GRID * S_GRID], tf.float32),
    "y": tf.io.FixedLenFeature([C_OUT * S_GRID * S_GRID], tf.float32),
    "x_m": tf.io.FixedLenFeature([1], tf.float32),
    "y_m": tf.io.FixedLenFeature([1], tf.float32),
    "sample_id": tf.io.FixedLenFeature([1], tf.string),
    "city": tf.io.FixedLenFeature([1], tf.string),
    "split": tf.io.FixedLenFeature([1], tf.string),
}

def parse_spatial_fn(rec):
    c, h = S_GRID // 2, PX // 2
    e = tf.io.parse_single_example(rec, spec_spatial)
    x = tf.transpose(tf.reshape(e["x"], [C_IN, S_GRID, S_GRID]), [1, 2, 0])[c - h:c + h + 1, c - h:c + h + 1]
    y = tf.transpose(tf.reshape(e["y"], [C_OUT, S_GRID, S_GRID]), [1, 2, 0])[c - h:c + h + 1, c - h:c + h + 1]
    valid = tf.cast(y[..., 0] != NODATA, tf.float32)
    bldg_h = tf.where(y[..., 1] == NODATA, 0.0, tf.maximum(y[..., 1], 0.0))
    return x, bldg_h, valid, e["x_m"][0], e["y_m"][0], e["sample_id"][0], e["city"][0]

tfr_folders = [TRAIN_DIR] + (["patch_dataset_test"] if INCLUDE_TEST_FOLDER and os.path.exists("patch_dataset_test") else [])
all_files = sorted([f for fld in tfr_folders for f in glob.glob(f"{fld}/{PATTERN}")])

ds_city = (
    tf.data.TFRecordDataset(all_files, compression_type="GZIP", num_parallel_reads=tf.data.AUTOTUNE)
    .map(parse_spatial_fn, num_parallel_calls=tf.data.AUTOTUNE)
    .filter(lambda *a: tf.equal(a[6], TARGET_CITY))
)

batches = list(ds_city.batch(512).prefetch(1).as_numpy_iterator())
assert len(batches) > 0, f"Nenhum patch encontrado para TARGET_CITY='{TARGET_CITY}'."
cols_c = [np.concatenate(c) for c in zip(*batches)]
city_data = dict(zip(["x", "h_map", "mask_px", "x_m", "y_m", "sid", "city"], cols_c))
city_data["sid"] = np.array([s.decode() for s in city_data["sid"]])
_, uniq_idx = np.unique(city_data["sid"], return_index=True)
city_data = {k: v[np.sort(uniq_idx)] for k, v in city_data.items()}

print(f"\nReconstruindo '{TARGET_CITY}' a partir de {len(city_data['sid'])} patches selecionados ({PX}x{PX} px)...")

X_city = prep(city_data["x"], mean, std)
p_bldg_c, h_cond_c, h_pred_c = predict_multitask(model, X_city, prob_thresh=best_th)

half = PX // 2
gx = np.round(city_data["x_m"] / SCALE_M).astype(np.int64)
gy = np.round(city_data["y_m"] / SCALE_M).astype(np.int64)

gx_min, gx_max = int(gx.min() - half), int(gx.max() + half)
gy_min, gy_max = int(gy.min() - half), int(gy.max() + half)
W_grid = gx_max - gx_min + 1
H_grid = gy_max - gy_min + 1

acc_true = np.zeros((H_grid, W_grid), dtype=np.float64)
acc_pred = np.zeros((H_grid, W_grid), dtype=np.float64)
acc_prob = np.zeros((H_grid, W_grid), dtype=np.float64)
acc_cnt  = np.zeros((H_grid, W_grid), dtype=np.float64)

for i in range(len(gx)):
    c0 = int(gx[i] - half - gx_min)
    r0 = int(gy_max - (gy[i] + half))
    m = city_data["mask_px"][i] > 0.5
    acc_true[r0:r0 + PX, c0:c0 + PX] += np.where(m, city_data["h_map"][i], 0.0)
    acc_pred[r0:r0 + PX, c0:c0 + PX] += np.where(m, h_pred_c[i], 0.0)
    acc_prob[r0:r0 + PX, c0:c0 + PX] += np.where(m, p_bldg_c[i], 0.0)
    acc_cnt[r0:r0 + PX, c0:c0 + PX]  += m.astype(np.float64)

has_px = acc_cnt > 0
map_true = np.where(has_px, acc_true / np.maximum(acc_cnt, 1.0), np.nan)
map_pred = np.where(has_px, acc_pred / np.maximum(acc_cnt, 1.0), np.nan)
map_prob = np.where(has_px, acc_prob / np.maximum(acc_cnt, 1.0), np.nan)
map_err  = map_pred - map_true

city_metrics = dense_metrics(city_data["h_map"], h_pred_c, city_data["mask_px"])
print(f"Grade reconstruída: {H_grid} x {W_grid} px ({H_grid * SCALE_M / 1000:.2f} km x {W_grid * SCALE_M / 1000:.2f} km)")
print(f"Métricas {TARGET_CITY} -> RMSE_px: {city_metrics['RMSE_px']:.3f} m | RMSE_bldg: {city_metrics['RMSE_bldg_px']:.3f} m | MAE: {city_metrics['MAE_px']:.3f} m | R2: {city_metrics['R2_px']:.3f}")

os.makedirs(cs.OUT / "multitask_experimentation", exist_ok=True)
npz_path = f"{cs.OUT}/multitask_experimentation/city_reconstruction_{TARGET_CITY}.npz"
np.savez_compressed(
    npz_path,
    map_true=map_true.astype(np.float32),
    map_pred=map_pred.astype(np.float32),
    map_prob=map_prob.astype(np.float32),
    map_err=map_err.astype(np.float32),
    extent_utm_m=np.array([gx_min * SCALE_M, (gx_max + 1) * SCALE_M, (gy_min - 1) * SCALE_M, gy_max * SCALE_M]),
)

vmax_h = float(np.nanpercentile(np.concatenate([map_true[has_px], map_pred[has_px]]), 99.0))
vmax_h = max(vmax_h, 15.0)
extent_km = [0.0, W_grid * SCALE_M / 1000.0, 0.0, H_grid * SCALE_M / 1000.0]

cmap_h = plt.cm.turbo.copy()
cmap_h.set_bad(color="#141414")
cmap_p = plt.cm.magma.copy()
cmap_p.set_bad(color="#141414")
cmap_e = plt.cm.RdBu_r.copy()
cmap_e.set_bad(color="#141414")

fig, axes = plt.subplots(2, 2, figsize=(16, 14), dpi=140, constrained_layout=True)

im0 = axes[0, 0].imshow(map_true, cmap=cmap_h, vmin=0, vmax=vmax_h, extent=extent_km)
axes[0, 0].set_title(f"{TARGET_CITY.replace('_', ' ').title()} — LiDAR Real (Ground Truth)", fontsize=12, fontweight="bold")
fig.colorbar(im0, ax=axes[0, 0], fraction=0.046, pad=0.03, label="Altura (m)")

im1 = axes[0, 1].imshow(map_pred, cmap=cmap_h, vmin=0, vmax=vmax_h, extent=extent_km)
axes[0, 1].set_title(f"Reconstrução Multi-Task U-Net (RMSE={city_metrics['RMSE_px']:.2f} m | R²={city_metrics['R2_px']:.2f})", fontsize=12, fontweight="bold")
fig.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.03, label="Altura Predita (m)")

im2 = axes[1, 0].imshow(map_prob, cmap=cmap_p, vmin=0, vmax=1, extent=extent_km)
axes[1, 0].set_title("Cabeça 1: Probabilidade de Footprint (p_bldg)", fontsize=12, fontweight="bold")
fig.colorbar(im2, ax=axes[1, 0], fraction=0.046, pad=0.03, label="Probabilidade")

im3 = axes[1, 1].imshow(map_err, cmap=cmap_e, vmin=-15, vmax=15, extent=extent_km)
axes[1, 1].set_title(f"Erro Espacial (Predito − LiDAR | Viés={city_metrics['bias_px']:+.2f} m)", fontsize=12, fontweight="bold")
fig.colorbar(im3, ax=axes[1, 1], fraction=0.046, pad=0.03, label="Erro (m)")

for ax in axes.flat:
    ax.set_xlabel("Leste (km)")
    ax.set_ylabel("Norte (km)")

fig_path = f"{cs.OUT}/multitask_experimentation/city_reconstruction_{TARGET_CITY}.png"
plt.savefig(fig_path, dpi=180)
plt.show()
print("Arquivos salvos:", fig_path, "e", npz_path)

In [ ]:
# ==============================================================================
# 6. ZOOM REGIONAL NA RECONSTRUCAO ESPACIAL (x: 0..15 km | y: 20..30 km)
# ==============================================================================
X_ZOOM_KM = (0.0, 15.0)
Y_ZOOM_KM = (20.0, 30.0)

# Conversao de km -> indices de linha/coluna (lembrando que r=0 e o topo/norte = H_grid)
c_start = max(0, int(np.floor(X_ZOOM_KM[0] * 1000.0 / SCALE_M)))
c_end   = min(W_grid, int(np.ceil(X_ZOOM_KM[1] * 1000.0 / SCALE_M)))

r_start = max(0, int(np.floor(H_grid - (Y_ZOOM_KM[1] * 1000.0 / SCALE_M))))
r_end   = min(H_grid, int(np.ceil(H_grid - (Y_ZOOM_KM[0] * 1000.0 / SCALE_M))))

assert (c_end > c_start) and (r_end > r_start), (
    f"Janela de zoom fora dos limites da grade ({W_grid * SCALE_M / 1000:.2f} km x {H_grid * SCALE_M / 1000:.2f} km)."
)

z_true = map_true[r_start:r_end, c_start:c_end]
z_pred = map_pred[r_start:r_end, c_start:c_end]
z_prob = map_prob[r_start:r_end, c_start:c_end]
z_err  = map_err[r_start:r_end, c_start:c_end]
z_valid = ~np.isnan(z_true)

# Metricas locais exclusivas da area do zoom
zoom_metrics = dense_metrics(z_true[None, ...], z_pred[None, ...], z_valid[None, ...].astype(np.float32))
print(
    f"\nZoom {TARGET_CITY} [x={X_ZOOM_KM[0]}..{X_ZOOM_KM[1]} km, y={Y_ZOOM_KM[0]}..{Y_ZOOM_KM[1]} km] -> "
    f"RMSE_px: {zoom_metrics['RMSE_px']:.3f} m | RMSE_bldg: {zoom_metrics['RMSE_bldg_px']:.3f} m | "
    f"MAE: {zoom_metrics['MAE_px']:.3f} m | Viés: {zoom_metrics['bias_px']:+.3f} m | R2: {zoom_metrics['R2_px']:.3f}"
)

extent_zoom_km = [
    c_start * SCALE_M / 1000.0,
    c_end * SCALE_M / 1000.0,
    (H_grid - r_end) * SCALE_M / 1000.0,
    (H_grid - r_start) * SCALE_M / 1000.0,
]

vmax_z = float(np.nanpercentile(np.concatenate([z_true[z_valid], z_pred[z_valid]]), 99.0)) if z_valid.any() else 30.0
vmax_z = max(vmax_z, 15.0)

fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=150, constrained_layout=True)

im0 = axes[0, 0].imshow(z_true, cmap=cmap_h, vmin=0, vmax=vmax_z, extent=extent_zoom_km)
axes[0, 0].set_title(
    f"{TARGET_CITY.replace('_', ' ').title()} (Zoom) — LiDAR Real",
    fontsize=12, fontweight="bold"
)
fig.colorbar(im0, ax=axes[0, 0], fraction=0.046, pad=0.03, label="Altura (m)")

im1 = axes[0, 1].imshow(z_pred, cmap=cmap_h, vmin=0, vmax=vmax_z, extent=extent_zoom_km)
axes[0, 1].set_title(
    f"Multi-Task U-Net (Zoom: RMSE={zoom_metrics['RMSE_px']:.2f} m | R²={zoom_metrics['R2_px']:.2f})",
    fontsize=12, fontweight="bold"
)
fig.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.03, label="Altura Predita (m)")

im2 = axes[1, 0].imshow(z_prob, cmap=cmap_p, vmin=0, vmax=1, extent=extent_zoom_km)
axes[1, 0].set_title("Cabeça 1: Probabilidade de Footprint (Zoom)", fontsize=12, fontweight="bold")
fig.colorbar(im2, ax=axes[1, 0], fraction=0.046, pad=0.03, label="Probabilidade")

im3 = axes[1, 1].imshow(z_err, cmap=cmap_e, vmin=-15, vmax=15, extent=extent_zoom_km)
axes[1, 1].set_title(
    f"Erro Espacial no Zoom (Predito − LiDAR | Viés={zoom_metrics['bias_px']:+.2f} m)",
    fontsize=12, fontweight="bold"
)
fig.colorbar(im3, ax=axes[1, 1], fraction=0.046, pad=0.03, label="Erro (m)")

for ax in axes.flat:
    ax.set_xlabel("Leste (km)")
    ax.set_ylabel("Norte (km)")

zoom_fig_path = f"{cs.OUT}/multitask_experimentation/city_reconstruction_{TARGET_CITY}_zoom.png"
plt.savefig(zoom_fig_path, dpi=180)
plt.show()
print("Figura de zoom salva em:", zoom_fig_path)